# Institutional Trading Desk

A causal, daily-bar research workflow for single-name monitoring, risk framing, scenario design, cross-sectional triage, and DPO feature hand-off.

**Operating contract**

- Uses adjusted, completed daily OHLCV bars from one immutable download per ticker.
- Publishes swing structure only after the right-hand confirmation window has elapsed.
- Treats supply, demand, AVWAP, and trade levels as transparent research references—not probabilities or trade recommendations.
- Renders the interactive Plotly dashboard inline in this notebook. It does not create an HTML file.
- Keeps feature preprocessing out of the full sample: fit imputers, clipping thresholds, and scalers on each training fold only.


## 1. Environment

Run this once in a fresh notebook runtime. The local src directory remains the source of truth for the desk calculations.

In [ ]:
%pip install -q yfinance pandas numpy plotly

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break
else:
    raise RuntimeError("Run the notebook from the FINRL repository or its notebooks directory.")

from finrl.research.institutional_desk import (
    DeskConfig,
    analyze_ticker,
    create_desk_chart,
    factor_panel,
    rank_universe,
    scan_universe,
    scenario_frame,
    snapshot_frame,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

## 2. Desk configuration

The default single-name is Zscaler (ZS). The universe is intentionally small so the notebook remains responsive; expand it only after validating data availability and execution time.

In [ ]:
TICKER = "NFLX"
PERIOD = "5y"
CHART_BARS = 220
UNIVERSE = ["ZS", "CRWD", "PANW", "OKTA", "MSFT", "NVDA", "SPY", "QQQ"]

DESK_CONFIG = DeskConfig(
    period=PERIOD,
    chart_bars=CHART_BARS,
    swing_left=3,
    swing_right=3,
    max_trigger_distance_atr=2.5,
    max_stop_distance_atr=2.0,
    max_risk_fraction=0.08,
)

## 3. Single-name monitor

This is the observable desk state as of the latest completed bar. It separates descriptive market state from the scenario book shown later.

In [ ]:
analysis = analyze_ticker(TICKER, DESK_CONFIG)
snapshot = analysis.snapshot

headline = pd.DataFrame(
    {
        "as_of": [snapshot.as_of.date()],
        "close": [snapshot.last_close],
        "trend": [snapshot.primary_trend],
        "confirmed_structure": [snapshot.market_structure],
        "avwap_state": [snapshot.avwap_state],
        "price_volume_state": [snapshot.price_volume_state],
    },
    index=[snapshot.ticker],
)
display(headline)
display(snapshot_frame(snapshot).T.rename(columns={0: "value"}))

## 4. Data and preprocessing control

The desk validates chronological order, duplicate dates, numeric OHLCV fields, positive prices, non-negative volume, OHLC geometry, and sufficient warm-up history before calculating features. The table below makes remaining warm-up missingness explicit.

In [ ]:
features = analysis.history
health_columns = [
    "close", "atr", "ema_fast", "ema_mid", "ema_slow",
    "momentum_21d", "momentum_126_21d", "near_52w_high",
    "realized_vol_20", "downside_vol_60", "max_drawdown_126",
    "volume_z_20", "log_adv_20", "amihud_20",
    "structure_score", "swing_avwap", "stress_avwap",
]

data_health = pd.DataFrame(
    {
        "dtype": features[health_columns].dtypes.astype(str),
        "missing_pct_full_history": features[health_columns].isna().mean().mul(100),
        "latest_is_finite": [
            bool(np.isfinite(pd.to_numeric(features[column].iloc[-1], errors="coerce")))
            for column in health_columns
        ],
        "latest_value": [features[column].iloc[-1] for column in health_columns],
    },
    index=health_columns,
)
display(data_health.round(4))

assert features.index.is_monotonic_increasing
assert not features.index.has_duplicates
required_latest = data_health.drop(index=["stress_avwap"])
assert required_latest["latest_is_finite"].all(), "Latest required state contains an unusable feature."

## 5. Factor book

The panel is grouped into trend, confirmed market structure, momentum, price location, risk, liquidity, volume, and anchored-price references. These are OHLCV-derived institutional-style proxies; they are not substitutes for order-book, options, borrow, fundamentals, or proprietary flow data.

In [ ]:
display(factor_panel(snapshot))

## 6. Scenario and risk book

Each side has an explicit trigger, invalidation stop, two R-multiple targets, trigger distance, and eligibility gate. Eligible means the deterministic desk rules pass; it does not mean confidence, probability of success, or permission to trade.

In [ ]:
scenarios = scenario_frame(snapshot)
scenario_columns = [
    "direction", "status", "eligible", "trigger", "stop",
    "target_1", "target_2", "risk_per_share", "risk_fraction",
    "reward_risk_1", "reward_risk_2", "trigger_distance_atr",
    "thesis", "invalidation",
]
display(scenarios[scenario_columns])

## 7. Inline desk dashboard

The first panel combines price, EMAs, confirmed pivots, causal AVWAPs, clustered supply/demand references, and scenario levels. The second shows participation. The third shows realized volatility and rolling drawdown. Confirmed pivot markers appear on the date the desk could first know them, not retroactively on the pivot date.

In [ ]:
desk_figure = create_desk_chart(analysis, DESK_CONFIG, show=True)

## 8. Cross-sectional blotter

The research-priority score equally weights transparent momentum, structure, risk, and liquidity percentile sleeves. It is a triage queue for analyst attention—not a forecast, alpha claim, or portfolio weight.

In [ ]:
universe_scan = scan_universe(UNIVERSE, DESK_CONFIG)
ranked_universe = rank_universe(universe_scan)

blotter_columns = [
    "ticker", "as_of", "last_close", "primary_trend", "market_structure",
    "momentum_rank", "structure_rank", "risk_rank", "liquidity_rank",
    "research_priority", "realized_vol_20", "max_drawdown_126",
]
if "error" in ranked_universe.columns:
    blotter_columns.append("error")
display(ranked_universe.reindex(columns=blotter_columns).round(4))

## 9. DPO state hand-off

The matrix below is the unscaled, as-of-close state candidate. For a next-session execution model, state dated t must drive the action executed at t+1. In walk-forward training, fit median imputation, clipping, and robust scaling on the training window only; freeze those parameters for validation/test; preserve missingness masks; and never smooth HMM regimes with future observations.

In [ ]:
DPO_STATE_COLUMNS = [
    "momentum_21d", "momentum_126_21d", "near_52w_high",
    "realized_vol_20", "downside_vol_60", "max_drawdown_126", "atr",
    "volume_z_20", "log_adv_20", "amihud_20", "close_location",
    "trend_score", "structure_score", "bars_since_swing_high",
    "bars_since_swing_low", "swing_avwap_distance_atr", "stress_score",
]

dpo_state_asof_close = (
    features[DPO_STATE_COLUMNS]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)
dpo_missing_mask = dpo_state_asof_close.isna().add_suffix("__missing")

display(dpo_state_asof_close.tail())
display(dpo_missing_mask.tail())

print(
    f"DPO hand-off: {dpo_state_asof_close.shape[1]} raw state features; "
    "preprocessing must be fit inside each walk-forward training fold."
)

## Desk interpretation checklist

1. Confirm the as-of date and data-health assertions before reading signals.
2. Use trend and confirmed structure as state, not as a standalone entry rule.
3. Read demand/supply zones as clusters of confirmed pivot references within an ATR tolerance. A larger touch count means more repeated references, not guaranteed support or resistance.
4. Treat AVWAP as a daily-bar approximation of participant cost basis. The swing anchor becomes usable only when the pivot is confirmed; the stress anchor is chosen causally from a rolling window.
5. Reject scenarios that fail risk or distance gates, then apply portfolio-level exposure, liquidity, turnover, and transaction-cost constraints outside this notebook.
6. Validate every feature and composite score out of sample before allowing it to influence DPO rewards or allocations.
